In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import nltk
from nltk.corpus import stopwords
import string

# Configurações Iniciais
nltk.download('punkt')
nltk.download('stopwords')
stops = set(stopwords.words('portuguese'))

def pipeline_limpeza(texto):
    texto = texto.lower().translate(str.maketrans('', '', string.punctuation))
    tokens = nltk.word_tokenize(texto)
    return " ".join([w for w in tokens if w not in stops])

# --- DATASET DE TREINO PARA OS EXERCÍCIOS ---
faq_suporte = [
    "Como posso redefinir minha senha de acesso?",
    "Onde encontro o boleto para pagamento da fatura?",
    "Meu pedido está atrasado, quero saber o status da entrega.",
    "Não consigo fazer login no aplicativo, aparece erro 404.",
    "Quero cancelar minha assinatura e pedir estorno.",
    "Como atualizar meu endereço de entrega no cadastro?"
]

# ---------------------------------------------------------
# EXERCÍCIO 1: O "PESO" DA DIMENSIONALIDADE
# Objetivo: Entender como o vocabulário dita o tamanho da matriz.
# ---------------------------------------------------------
def exercicio_01():
    print("\n>>> EXERCÍCIO 1: ANÁLISE DE DIMENSIONALIDADE")

    # Tarefa: Adicione 2 frases novas e bem diferentes ao FAQ e veja o vocabulário crescer.
    faq_expandido = faq_suporte + [
        "quais as formas de pagamento aceitas",
        "como falar com um atendente humano"
    ]

    cv = CountVectorizer()
    matriz = cv.fit_transform(faq_expandido)

    print(f"Total de Documentos (Linhas): {matriz.shape[0]}")
    print(f"Total de Palavras Únicas (Colunas/Features): {matriz.shape[1]}")
    print(f"Vocabulário Gerado: {list(cv.get_feature_names_out()[:10])}... (e mais)")

    # PERGUNTA PARA O ALUNO: Se tivéssemos 1 milhão de frases, o que aconteceria com a memória do PC?

# ---------------------------------------------------------
# EXERCÍCIO 2: A SENSIBILIDADE DO TF-IDF
# Objetivo: Provar que palavras comuns perdem valor estatístico.
# ---------------------------------------------------------
def exercicio_02():
    print("\n>>> EXERCÍCIO 2: O FILTRO TF-IDF NA PRÁTICA")

    # Vamos injetar a palavra "ajuda" em TODAS as frases.
    faq_comum = [frase + " ajuda" for frase in faq_suporte]

    tfidf = TfidfVectorizer()
    X_tfidf = tfidf.fit_transform(faq_comum)
    df_tfidf = pd.DataFrame(X_tfidf.toarray(), columns=tfidf.get_feature_names_out())

    # Vamos verificar o peso da palavra 'ajuda' (que está em todas)
    # versus a palavra 'senha' (que é rara).
    palavra_comum = 'ajuda'
    palavra_rara = 'senha'

    print(f"Peso TF-IDF da palavra '{palavra_comum}': {df_tfidf[palavra_comum].mean():.4f}")
    print(f"Peso TF-IDF da palavra '{palavra_rara}':  {df_tfidf[palavra_rara].max():.4f}")

    # CONCLUSÃO: O TF-IDF penaliza 'ajuda' porque ela não ajuda a distinguir uma frase da outra.

# ---------------------------------------------------------
# EXERCÍCIO 3: BUSCADOR DE FAQ (SIMILARIDADE DE COSSENO)
# Objetivo: Criar a lógica de um motor de busca semântico simples.
# ---------------------------------------------------------
def exercicio_03():
    print("\n>>> EXERCÍCIO 3: MOTOR DE BUSCA POR SIMILARIDADE")

    # 1. Vetorizamos o nosso FAQ original
    tfidf = TfidfVectorizer()
    X_faq = tfidf.fit_transform(faq_suporte)

    # 2. Pergunta do usuário (O que o bot acabou de receber)
    query_usuario = "esqueci minha senha e não consigo entrar"
    print(f"Pergunta do Usuário: '{query_usuario}'")

    # 3. Vetorizamos a pergunta usando o MESMO vocabulário do FAQ
    X_query = tfidf.transform([query_usuario])

    # 4. Cálculo da Similaridade de Cosseno (compara a query com todas as frases do FAQ)
    # Retorna uma lista de scores de 0 (nada a ver) a 1 (idêntico)
    similaridades = cosine_similarity(X_query, X_faq).flatten()

    # 5. Encontrar a melhor resposta
    indice_melhor = np.argmax(similaridades)
    score_maximo = similaridades[indice_melhor]

    print("-" * 30)
    print(f"Melhor Resposta Encontrada: {faq_suporte[indice_melhor]}")
    print(f"Confiança do Bot: {score_maximo * 100:.2f}%")
    print("-" * 30)

if __name__ == "__main__":
    exercicio_01()
    exercicio_02()
    exercicio_03()

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.



>>> EXERCÍCIO 1: ANÁLISE DE DIMENSIONALIDADE
Total de Documentos (Linhas): 8
Total de Palavras Únicas (Colunas/Features): 47
Vocabulário Gerado: ['404', 'aceitas', 'acesso', 'aparece', 'aplicativo', 'as', 'assinatura', 'atendente', 'atrasado', 'atualizar']... (e mais)

>>> EXERCÍCIO 2: O FILTRO TF-IDF NA PRÁTICA
Peso TF-IDF da palavra 'ajuda': 0.1694
Peso TF-IDF da palavra 'senha':  0.4011

>>> EXERCÍCIO 3: MOTOR DE BUSCA POR SIMILARIDADE
Pergunta do Usuário: 'esqueci minha senha e não consigo entrar'
------------------------------
Melhor Resposta Encontrada: Como posso redefinir minha senha de acesso?
Confiança do Bot: 35.58%
------------------------------


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [4]:
# ==========================
# 1. IMPORTAÇÕES
# ==========================
import re
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

# ==========================
# 2. BASE DE DADOS (INTENÇÕES)
# ==========================
frases = [
    "Quero cancelar meu pedido",
    "Preciso cancelar a compra",
    "Como faço para cancelar?",

    "Produto veio com defeito",
    "Minha entrega atrasou",
    "Estou insatisfeito com o produto",

    "Qual o status do meu pedido?",
    "Quando chega minha entrega?",
    "Quero saber o prazo de entrega"
]

labels = [
    "cancelamento", "cancelamento", "cancelamento",
    "reclamacao", "reclamacao", "reclamacao",
    "consulta", "consulta", "consulta"
]

# ==========================
# 3. STOPWORDS
# ==========================
stopwords = ["o", "a", "de", "do", "da", "meu", "minha", "com", "para", "e", "como"]

# ==========================
# 4. FUNÇÃO DE PRÉ-PROCESSAMENTO
# ==========================
def preprocessar_texto(texto):
    texto = texto.lower()  # normalização
    texto = re.sub(r'[^a-zà-ú\s]', '', texto)  # limpeza
    tokens = texto.split()  # tokenização
    tokens = [t for t in tokens if t not in stopwords]  # remoção de stopwords
    return " ".join(tokens)  # retorna string novamente (necessário para vetorizar)

# Aplica o pré-processamento em todas as frases
corpus = [preprocessar_texto(f) for f in frases]

# ==========================
# 5. VETORIZAÇÃO (BAG OF WORDS)
# ==========================
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(corpus)

# ==========================
# 6. DIVISÃO TREINO / TESTE
# ==========================
X_train, X_test, y_train, y_test = train_test_split(
    X, labels, test_size=0.3, random_state=42
)

# ==========================
# 7. TREINAMENTO DO MODELO
# ==========================
modelo = MultinomialNB()
modelo.fit(X_train, y_train)

# ==========================
# 8. AVALIAÇÃO DO MODELO
# ==========================
y_pred = modelo.predict(X_test)

acuracia = accuracy_score(y_test, y_pred)
print(f"Acurácia do modelo: {acuracia:.2f}")

# ==========================
# 9. SIMULAÇÃO DE CHATBOT
# ==========================
print("\n=== CHATBOT INTELIGENTE ===")
print("Digite 'sair' para encerrar\n")

while True:
    entrada = input("Usuário: ")

    if entrada.lower() == "sair":
        print("Chatbot: Encerrando atendimento...")
        break

    # pré-processa entrada
    entrada_proc = preprocessar_texto(entrada)

    # transforma em vetor
    entrada_vec = vectorizer.transform([entrada_proc])

    # prediz intenção
    intencao = modelo.predict(entrada_vec)[0]

    # respostas simples baseadas na intenção
    if intencao == "cancelamento":
        resposta = "Posso te ajudar a cancelar seu pedido."
    elif intencao == "reclamacao":
        resposta = "Sinto muito pelo problema. Vamos resolver isso."
    elif intencao == "consulta":
        resposta = "Vou verificar essa informação para você."
    else:
        resposta = "Não entendi sua solicitação."

    print(f"Chatbot ({intencao}): {resposta}\n")

Acurácia do modelo: 0.67

=== CHATBOT INTELIGENTE ===
Digite 'sair' para encerrar

Usuário: quero cancelar um pedido
Chatbot (cancelamento): Posso te ajudar a cancelar seu pedido.

Usuário: como consulto um pedido?
Chatbot (cancelamento): Posso te ajudar a cancelar seu pedido.

Usuário: quero fazer uma reclamação
Chatbot (cancelamento): Posso te ajudar a cancelar seu pedido.

Usuário: sair
Chatbot: Encerrando atendimento...


In [5]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
import nltk
from nltk.corpus import stopwords
import string

# (pré-processamento simplificado)
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab') # Download the missing 'punkt_tab' resource
stops = set(stopwords.words('portuguese'))

def limpar_simplificado(texto):
    texto = texto.lower().translate(str.maketrans('', '', string.punctuation))
    tokens = nltk.word_tokenize(texto, language='portuguese') # Explicitly specify Portuguese
    tokens_limpos = [w for w in tokens if w not in stops and w.isalnum()]
    return " ".join(tokens_limpos)

# 1. DADOS DE ENTRADA (Dataset Grande de Logs Não Rotulados)
# Imagine que estas são 15 mensagens novas que o bot recebeu
mensagens_logs = [
    "meu boleto nao chegou", "queria pagar com pix", "erro no login nao entra",
    "atraso no pedido entrega demora", "cartao recusado pagamento falhou",
    "login bloqueado senha invalida", "frete muito caro para o meu cep",
    "quero cancelar a compra", "boleto vencido como pagar",
    "pix nao funciona erro no app", "pedido nao foi entregue prazo venceu",
    "cancelamento de pedido como fazer", "valor do frete cep 01001",
    "senha esquecida nao consigo login", "estorno do pagamento cancelado"
]

df = pd.DataFrame({'mensagem_bruta': mensagens_logs})
df['mensagem_limpa'] = df['mensagem_bruta'].apply(limpar_simplificado)

# 2. VETORIZAÇÃO
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df['mensagem_limpa'])

# 3. EXERCÍCIO: CLUSTERIZAÇÃO (K-Means)
# Defina o número de clusters (grupos) que você deseja encontrar
# Testem com 3, 4 e 5 clusters
N_CLUSTERS = 4

print(f"--- Executando K-Means para encontrar {N_CLUSTERS} grupos ---")
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
kmeans.fit(X)

# Adiciona o rótulo do cluster de volta ao DataFrame
df['cluster'] = kmeans.labels_

# 4. ANÁLISE DOS RESULTADOS
# Vamos imprimir as mensagens de cada cluster para ver se o agrupamento faz sentido
for i in range(N_CLUSTERS):
    print(f"\n[CLUSTER {i}] - Tópico Sugerido:")
    messages_in_cluster = df[df['cluster'] == i]['mensagem_bruta'].tolist()

    # Imprime as primeiras 4 mensagens do cluster
    for msg in messages_in_cluster[:4]:
        print(f"  - {msg}")

# OBSERVAÇÃO TÉCNICA:
# Provavelmente o K-Means agrupará mensagens sobre:
# Cluster X: Pagamentos (boleto, pix, cartao)
# Cluster Y: Logística/Entrega (atraso, pedido, frete, cep)
# Cluster Z: Acesso (login, senha, bloqueado)
# Cluster W: Cancelamento (cancelar, cancelado, estorno)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


--- Executando K-Means para encontrar 4 grupos ---

[CLUSTER 0] - Tópico Sugerido:
  - erro no login nao entra
  - login bloqueado senha invalida
  - senha esquecida nao consigo login

[CLUSTER 1] - Tópico Sugerido:
  - frete muito caro para o meu cep
  - valor do frete cep 01001

[CLUSTER 2] - Tópico Sugerido:
  - meu boleto nao chegou
  - queria pagar com pix
  - cartao recusado pagamento falhou
  - quero cancelar a compra

[CLUSTER 3] - Tópico Sugerido:
  - atraso no pedido entrega demora
  - pedido nao foi entregue prazo venceu
  - cancelamento de pedido como fazer
